# 05b — HOME consensus tiers + adaptive WORK audit

Stage 05 showed stronger convergence for HOME than OFFICE. This notebook reuses its private caches and does not change production inference.

```text
HOME → consensus tiers
WORK → sliding-window secondary-anchor audit
```


In [ ]:
from pathlib import Path
import importlib.util, os, subprocess, sys
REPO_URL='https://github.com/tanh1c/geolife.git'
REPO_BRANCH=os.environ.get('GEOLIFE_REPO_BRANCH','main')
REPO_DIR=Path(os.environ.get('GEOLIFE_REPO_DIR','/tmp/geolife'))
VOLUME_ROOT=Path('/mnt/geolife-data')
CACHE_05=VOLUME_ROOT/'cache'/'05_home_office_reliability_validation'
CACHE_05B=VOLUME_ROOT/'cache'/'05b_home_consensus_adaptive_work'
CACHE_05B.mkdir(parents=True,exist_ok=True)
def run(cmd,cwd=None): subprocess.run(cmd,cwd=cwd,check=True)
if (REPO_DIR/'.git').exists():
    current=subprocess.check_output(['git','-C',str(REPO_DIR),'branch','--show-current'],text=True).strip()
    if current!=REPO_BRANCH: run(['git','-C',str(REPO_DIR),'checkout',REPO_BRANCH])
    if os.environ.get('GEOLIFE_REFRESH_REPO','0')=='1' or not (REPO_DIR/'analysis'/'05b_home_consensus_adaptive_work.py').exists():
        run(['git','-C',str(REPO_DIR),'fetch','origin',REPO_BRANCH])
        run(['git','-C',str(REPO_DIR),'reset','--hard',f'origin/{REPO_BRANCH}'])
else:
    run(['git','clone','--depth','1','--branch',REPO_BRANCH,REPO_URL,str(REPO_DIR)])
for p in (REPO_DIR,REPO_DIR/'src'):
    if str(p) not in sys.path: sys.path.insert(0,str(p))
try:
    import numpy as np, pandas as pd
    from geolife.model import HomeOfficeConfig, build_semantic_locations
except ModuleNotFoundError:
    run([sys.executable,'-m','pip','install','-q','-e','.'],cwd=REPO_DIR)
    import numpy as np, pandas as pd
    from geolife.model import HomeOfficeConfig, build_semantic_locations
spec=importlib.util.spec_from_file_location('stage05b',REPO_DIR/'analysis'/'05b_home_consensus_adaptive_work.py')
stage05b=importlib.util.module_from_spec(spec); sys.modules[spec.name]=stage05b; spec.loader.exec_module(stage05b)
print('Ready', CACHE_05B)


## 1. HOME consensus tiers

HIGH = ≥2 methods agree + split + held-out top-1 + dropout evidence. MEDIUM = ≥2 methods agree + at least 2/3 reliability axes. UNCERTAIN = otherwise. Tiers are audit summaries, not calibrated probabilities.


In [ ]:
paths={
 'assignments':CACHE_05/'assignments_private.pkl',
 'split':CACHE_05/'split_half_details_private.pkl',
 'holdout':CACHE_05/'holdout_details_private.pkl',
 'dropout':CACHE_05/'dropout_details_private.pkl'}
missing=[str(p) for p in paths.values() if not p.exists()]
if missing: raise FileNotFoundError('Run notebook 05 first:\n'+'\n'.join(missing))
assignments=pd.read_pickle(paths['assignments'])
split_details=pd.read_pickle(paths['split'])
holdout_details=pd.read_pickle(paths['holdout'])
dropout_details=pd.read_pickle(paths['dropout'])
display(pd.DataFrame([stage05b.synthetic_self_check()]))
home_evidence=stage05b.build_home_consensus(assignments,split_details,holdout_details,dropout_details)
home_winners=stage05b.home_consensus_winners(home_evidence)
home_summary=stage05b.summarize_home_consensus(home_evidence)
display(home_summary['tier_summary']); display(home_summary['production_summary']); display(home_summary['expansion_summary'])
home_evidence.to_pickle(CACHE_05B/'home_consensus_private.pkl')
for name,frame in home_summary.items(): frame.to_csv(CACHE_05B/f'home_{name}.csv',index=False)


## 2. Adaptive secondary-anchor audit

For HIGH/MEDIUM HOME users, exclude HOME and rank recurring secondary anchors in sliding windows. Selection does not use a fixed 09–17 clock window; arrival-hour concentration is measured after selection.


In [ ]:
def find_stay_cache():
    candidates=[VOLUME_ROOT/'cache'/'03a_user_behavior_deep_dive'/'stays_baseline_v1.pkl',VOLUME_ROOT/'artifacts'/'03a'/'stays_baseline_v1.pkl',REPO_DIR/'artifacts'/'03a'/'stays_baseline_v1.pkl']
    for p in candidates:
        if p.exists(): return p
    matches=sorted(VOLUME_ROOT.glob('**/stays_baseline_v1.pkl'))
    if matches: return matches[0]
    raise FileNotFoundError('Cannot find stays_baseline_v1.pkl')
stays=pd.read_pickle(find_stay_cache())
cfg=HomeOfficeConfig(clustering_method='complete_link',location_max_diameter_m=200.0)
semantic_stays,locations=build_semantic_locations(stays,config=cfg)
windows=stage05b.build_adaptive_work_windows(semantic_stays,home_evidence,window_days=42,step_days=14)
patterns=stage05b.summarize_adaptive_work_patterns(windows,stability_threshold=0.70)
compared=stage05b.compare_work_patterns_to_static(patterns,assignments)
pattern_summary=compared.groupby('window_pattern',as_index=False).agg(users=('user_id','nunique'),median_windows=('eligible_windows','median'),median_dominant_share=('dominant_window_share','median'),median_hour_concentration=('median_arrival_hour_concentration','median'),median_hour_shift_h=('median_dominant_hour_shift_h','median'))
display(pattern_summary)
windows.to_pickle(CACHE_05B/'adaptive_work_windows_42_private.pkl'); compared.to_pickle(CACHE_05B/'adaptive_work_patterns_42_private.pkl'); pattern_summary.to_csv(CACHE_05B/'adaptive_work_pattern_summary_42.csv',index=False)


## 3. Sensitivity and decision snapshot

Audit 28/42/56-day windows and 0.60/0.70/0.80 stability thresholds. `stable_secondary_anchor` is a behavioral pattern, not an OFFICE label.


In [ ]:
sensitivity,all_windows=stage05b.adaptive_work_sensitivity(semantic_stays,home_evidence,assignments,window_days_values=(28,42,56),stability_thresholds=(0.60,0.70,0.80),step_days=14)
display(sensitivity)
sensitivity.to_csv(CACHE_05B/'adaptive_work_sensitivity.csv',index=False); all_windows.to_pickle(CACHE_05B/'adaptive_work_all_windows_private.pkl')
expansion=home_winners.loc[home_winners['home_tier'].isin(['high','medium']) & ~home_winners['production_status'].eq('baseline_emitted')]
decision=pd.DataFrame([{'home_high':int(home_winners['home_tier'].eq('high').sum()),'home_medium':int(home_winners['home_tier'].eq('medium').sum()),'home_uncertain':int(home_winners['home_tier'].eq('uncertain').sum()),'home_high_medium_expansion':int(expansion['user_id'].nunique()),'work_stable_secondary_42d':int(compared['window_pattern'].eq('stable_secondary_anchor').sum()),'work_multi_anchor_42d':int(compared['window_pattern'].eq('multi_anchor').sum()),'work_unstable_42d':int(compared['window_pattern'].eq('unstable').sum()),'work_insufficient_42d':int(compared['window_pattern'].eq('insufficient').sum())}])
display(decision); decision.to_csv(CACHE_05B/'decision_snapshot.csv',index=False)
